# Initialization

In [1]:
# Laborers
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re

# Image workers
import cv2 as cv
import pytesseract
from PIL import Image, ImageOps

# Parallel processing
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from os import cpu_count

In [2]:
# Initialize pytesseract.
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Paths to images.
paths = list(Path("cq_results").iterdir())

# Functions

In [3]:
# OCR function.
def ocr(file):
    # Summon and crop.
    curr = Image.open(file).crop((0, 1370, 1080, 1450))
    curr = ImageOps.grayscale(curr)

    # OCR and append.
    return pytesseract.image_to_string(curr)

In [4]:
def parallel(paths):
    # Initialize parameters.
    results = []
    n_workers = cpu_count()- 1

    # Summon multithread.
    with ThreadPoolExecutor(max_workers = n_workers) as executor:
        futures = [executor.submit(ocr, p) for p in paths]

        # Load tqdm'ed processing.
        for future in tqdm(as_completed(futures), total = len(futures)):
            results.append(future.result())
            
    return results

# Processing

In [5]:
results = parallel(paths)

100%|██████████| 479/479 [00:19<00:00, 24.28it/s]


In [6]:
# Get names.
set(results)

{'',
 'Cyclops Dragon x1\n',
 'Demon King’s Memory x1\n',
 'Diamond Dragon x1\n',
 'Fiendish Memory x1\n',
 'Francis Drake x1\n',
 'Holy Sword Memory x1\n',
 'Jewelia x1\n',
 'Mysterious Memory x1\n',
 'Shogun’s Memory x1\n',
 'Tyrannical Memory x1\n',
 'War God’s Memory x1\n'}

In [7]:
# Set standradized drop names.
standard = {"Diamond Dragon x1\n": "5*",
            "Francis Drake x1\n" : "4* A",
            "Cyclops Dragon x1\n": "4* B",
            "Jewelia x1\n": "3*",
            # Magatama names are standard.
            "Demon King’s Memory x1\n|Tyrannical Memory x1\n|War God’s Memory x1\n": "5* M",
            "Holy Sword Memory x1\n|Fiendish Memory x1\n|Shogun’s Memory x1\n": "4* M",
            "Fiendish Memory x1\n|Mysterious Memory x1\n|Tender Memory x1\n": "3* M"}

In [8]:
# Replace drop names.
final = pd.Series(results).replace(standard, regex = True)
final.to_csv("drops_csv/cq_final.csv")